# Week 8: Multi-Agent Deal Hunting Framework

Autonomous system that scans online deals, estimates prices, and finds opportunities.

## Setup

In [ ]:
import os
import sys
import json
import re
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime
from dotenv import load_dotenv

week8_root = Path.cwd().parent if (Path.cwd().parent / 'agents').exists() else Path.cwd().parents[1]
sys.path.insert(0, str(week8_root))

load_dotenv(override=True)

print(f"OPENAI_API_KEY: {'SET' if os.getenv('OPENAI_API_KEY') else 'MISSING'}")

In [ ]:
from openai import OpenAI
import feedparser
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field
import gradio as gr

openai_client = OpenAI()

## Data Models

In [ ]:
class Deal(BaseModel):
    product_description: str
    price: float
    url: str

class Opportunity(BaseModel):
    deal: Deal
    estimate: float
    discount: float
    timestamp: str = Field(default_factory=lambda: datetime.now().strftime("%Y-%m-%d %H:%M"))

## Base Agent

In [ ]:
class Agent:
    COLORS = {'cyan': '\033[96m', 'green': '\033[92m', 'yellow': '\033[93m', 'blue': '\033[94m'}
    RESET = '\033[0m'
    
    name = "Agent"
    color = 'white'
    logs = []
    
    def log(self, message):
        color = self.COLORS.get(self.color, '')
        formatted = f"[{self.name}] {message}"
        print(f"{color}{formatted}{self.RESET}")
        Agent.logs.append(formatted)
    
    @classmethod
    def get_logs(cls):
        return "\n".join(cls.logs[-20:])
    
    @classmethod
    def clear_logs(cls):
        cls.logs = []

## Scanner Agent

In [ ]:
RSS_FEEDS = [
    "https://www.dealnews.com/c142/Electronics/?rss=1",
    "https://www.dealnews.com/c39/Computers/?rss=1",
]

class ScannerAgent(Agent):
    name = "Scanner"
    color = "cyan"
    
    def scan(self, max_deals=15):
        self.log(f"Scanning {len(RSS_FEEDS)} feeds...")
        deals = []
        
        for feed_url in RSS_FEEDS:
            feed = feedparser.parse(feed_url)
            for entry in feed.entries[:max_deals // len(RSS_FEEDS)]:
                soup = BeautifulSoup(entry.get('summary', ''), 'html.parser')
                deals.append({
                    'title': entry.get('title', '')[:150],
                    'summary': soup.get_text(strip=True)[:400],
                    'url': entry.get('links', [{}])[0].get('href', '')
                })
        
        self.log(f"Found {len(deals)} deals")
        return deals

## Frontier Agent

In [ ]:
class FrontierAgent(Agent):
    name = "Frontier"
    color = "green"
    
    def __init__(self):
        self.client = OpenAI()
    
    def estimate(self, description):
        self.log(f"Estimating: {description[:40]}...")
        
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": f"Estimate fair retail price in USD for: {description}. Reply with only a number."}],
            max_tokens=20
        )
        
        text = response.choices[0].message.content.replace('$', '').replace(',', '')
        match = re.search(r'\d+\.?\d*', text)
        return float(match.group()) if match else 0.0

## Messaging Agent

In [ ]:
class MessagingAgent(Agent):
    name = "Messenger"
    color = "yellow"
    
    def __init__(self):
        self.pushover_user = os.getenv('PUSHOVER_USER')
        self.pushover_token = os.getenv('PUSHOVER_TOKEN')
    
    def alert(self, opportunity):
        self.log(f"Alert: ${opportunity.discount:.0f} off {opportunity.deal.product_description[:30]}...")
        
        if self.pushover_user and self.pushover_token:
            requests.post("https://api.pushover.net/1/messages.json", data={
                "token": self.pushover_token,
                "user": self.pushover_user,
                "message": f"Deal: {opportunity.deal.product_description[:80]}\nPrice: ${opportunity.deal.price:.2f}\nValue: ${opportunity.estimate:.2f}",
                "url": opportunity.deal.url
            })

## Planning Agent

In [ ]:
class PlanningAgent(Agent):
    name = "Planner"
    color = "blue"
    
    def __init__(self):
        self.scanner = ScannerAgent()
        self.frontier = FrontierAgent()
        self.messenger = MessagingAgent()
    
    def plan(self, memory=None):
        self.log("Starting deal hunt...")
        memory = memory or []
        seen_urls = {opp.deal.url for opp in memory}
        new_opps = []
        
        raw_deals = self.scanner.scan(max_deals=15)
        
        for raw in raw_deals:
            if raw['url'] in seen_urls:
                continue
            
            deal = self._extract_deal(raw)
            if not deal or deal.price <= 0:
                continue
            
            estimate = self.frontier.estimate(deal.product_description)
            discount = estimate - deal.price
            
            if discount >= 15:
                opp = Opportunity(deal=deal, estimate=estimate, discount=discount)
                new_opps.append(opp)
                self.log(f"Found: ${discount:.0f} off - {deal.product_description[:40]}")
                self.messenger.alert(opp)
        
        self.log(f"Done: {len(new_opps)} opportunities")
        return new_opps
    
    def _extract_deal(self, raw):
        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": f"Extract from this deal. Return JSON only.\n\nTitle: {raw['title']}\nSummary: {raw['summary']}\n\nReturn: {{\"product_description\": \"...\", \"price\": 0.0}}"}],
                response_format={"type": "json_object"}
            )
            data = json.loads(response.choices[0].message.content)
            return Deal(product_description=data.get('product_description', raw['title']), price=float(data.get('price', 0)), url=raw['url'])
        except:
            return None

## Deal Agent Framework

In [ ]:
class DealAgentFramework:
    MEMORY_FILE = "memory.json"
    
    def __init__(self):
        self.memory = []
        if os.path.exists(self.MEMORY_FILE):
            with open(self.MEMORY_FILE) as f:
                self.memory = [Opportunity(**item) for item in json.load(f)]
        self.planner = PlanningAgent()
    
    def run(self):
        Agent.clear_logs()
        new_opps = self.planner.plan(self.memory)
        self.memory.extend(new_opps)
        with open(self.MEMORY_FILE, 'w') as f:
            json.dump([opp.model_dump() for opp in self.memory], f, indent=2)
        return self.memory
    
    def clear(self):
        self.memory = []
        if os.path.exists(self.MEMORY_FILE):
            os.remove(self.MEMORY_FILE)

In [ ]:
framework = DealAgentFramework()
opportunities = framework.run()
print(f"\nTotal: {len(opportunities)} opportunities")

## Gradio UI

In [ ]:
def create_ui():
    framework = DealAgentFramework()
    
    def get_table():
        return [[opp.deal.product_description[:60], f"${opp.deal.price:.2f}", f"${opp.estimate:.2f}", f"${opp.discount:.2f}", opp.deal.url] for opp in sorted(framework.memory, key=lambda x: x.discount, reverse=True)]
    
    def run_scan():
        framework.run()
        return get_table(), Agent.get_logs()
    
    def clear_all():
        framework.clear()
        Agent.clear_logs()
        return [], ""
    
    with gr.Blocks(title="Deal Hunter", theme=gr.themes.Soft()) as ui:
        gr.Markdown("# Deal Hunting AI\nScans deals, estimates prices, finds opportunities.")
        
        with gr.Row():
            scan_btn = gr.Button("Scan for Deals", variant="primary")
            clear_btn = gr.Button("Clear")
        
        table = gr.Dataframe(headers=["Product", "Price", "Value", "Savings", "Link"], value=get_table(), wrap=True)
        
        with gr.Accordion("Logs", open=False):
            logs = gr.Textbox(value="", lines=10, interactive=False)
        
        scan_btn.click(run_scan, outputs=[table, logs])
        clear_btn.click(clear_all, outputs=[table, logs])
    
    return ui

ui = create_ui()
ui.launch()